# Week 8 Assignment - Single Agent Pipeline

Celebal Technologies Internship
Submitted by: Sri Sharanya

## Objective

The goal of this assignment is to build a single-agent assistant that can look at a user's query, figure out what kind of request it is, and send it to the right tool - a calculator for math, a keyword extractor for text analysis, or just a general reply for everything else. Whatever happens, the agent always needs to return its answer as a dictionary with a "type" and "result" key, so the output format stays consistent no matter what path it takes.

## Problem Statement

A basic chatbot usually treats every query the same way, which isn't great because a math question, a request to pull keywords out of some text, and a random general question all need to be handled differently. This notebook is about building an agent that can tell these apart just from the query text and route it to the correct function automatically - without crashing even if someone gives it a weird or empty input.

## Import Required Libraries

I only needed two libraries for this:

- `json` - to pretty-print the agent's output nicely when testing.
- `re` - to pull out the actual expression or text from a query after removing the trigger word ("calculate" or "keywords"), which is more reliable than just slicing the string manually.

Both are built into Python so there's nothing extra to install.

In [ ]:
import json
import re

## Baseline Function Overview

These two functions were already given in the starter notebook, I didn't change them.

**calculator()** - takes an expression string and runs it through `eval()`. If it's not a valid expression (like it has letters in it, or is empty), it catches the exception and just returns "Error in calculation" instead of blowing up.

**extract_keywords()** - splits the input text into words, keeps only the ones longer than 4 letters (a simple way to skip small/common words), lowercases and de-duplicates them, and gives back up to 5.

In [ ]:
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [ ]:
def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## Agent Design

The starter code only had an empty `agent()` function with a TODO telling me to route based on whether the query has "calculate" or "keywords" in it. Here's how I thought about it:

- First, check that the query is actually valid - not None and not empty.
- Lowercase it just for the keyword check, so it doesn't matter if someone types "Calculate" or "CALCULATE".
- Whatever branch it goes into, it always has to come back as a dict with "type" and "result" - nothing else.

If neither keyword shows up, I just treat it as a general query. And if literally anything unexpected happens, there's a try/except around the whole function so it never crashes.

## Agent Workflow

Step by step, this is what `agent()` does:

1. Take in the query and check it isn't None/empty - if it is, return an error straight away.
2. Lowercase a copy of it to check for the keywords.
3. If "calculate" is in there - strip out the word "calculate" using regex so I'm left with just the expression, then pass that to `calculator()`. If the expression is empty after stripping, or `calculator()` comes back with its error string, I return a `type: "error"` response. Otherwise it's `type: "calculation"`.
4. If "keywords" is in there instead - same idea, strip out the word "keywords" and pass whatever's left to `extract_keywords()`, and return `type: "keywords"`. If there's nothing left after stripping, that's an error too.
5. If neither keyword matched at all, it's just a `type: "general"` response.
6. The whole thing sits inside a try/except so any surprise error still comes back as a clean `type: "error"` dict instead of crashing the notebook.

## Implementation

Here's the actual `agent()` function I wrote, following the logic above.

In [ ]:
def agent(query: str) -> dict:
    """Look at the query and route it to the right tool, always return a type/result dict."""
    try:
        if query is None or not isinstance(query, str) or query.strip() == "":
            return {
                "type": "error",
                "result": "Empty or invalid query provided."
            }

        query_lower = query.lower().strip()

        if "calculate" in query_lower:
            # pull out just the expression part, removing the word "calculate"
            expression = re.sub(r"calculate", "", query, flags=re.IGNORECASE).strip(" :")

            if expression == "":
                return {
                    "type": "error",
                    "result": "No mathematical expression found after 'calculate'."
                }

            calc_result = calculator(expression)

            if calc_result == "Error in calculation":
                return {
                    "type": "error",
                    "result": f"Could not evaluate expression: '{expression}'"
                }

            return {
                "type": "calculation",
                "result": calc_result
            }

        elif "keywords" in query_lower:
            # pull out the text part, removing the word "keywords"
            text = re.sub(r"keywords", "", query, flags=re.IGNORECASE).strip(" :")

            if text == "":
                return {
                    "type": "error",
                    "result": "No text found after 'keywords' to extract from."
                }

            keywords_result = extract_keywords(text)

            return {
                "type": "keywords",
                "result": keywords_result
            }

        else:
            return {
                "type": "general",
                "result": f"This is a general query and does not require a specific tool: '{query.strip()}'"
            }

    except Exception as e:
        return {
            "type": "error",
            "result": f"Unexpected error occurred: {str(e)}"
        }

## Validation Testing

I tested the agent against 12 different queries to make sure it covers all four response types - normal calculations, a broken calculation, a few keyword requests, an empty keyword request, and some general queries like greetings and random questions.

In [ ]:
test_queries = [
    "calculate 25+10",
    "calculate (12*8)-15",
    "calculate 10/2",
    "calculate abc+15",
    "keywords Artificial Intelligence is transforming Healthcare",
    "keywords Python is widely used in Data Science",
    "keywords Machine Learning enables predictive analytics",
    "keywords",
    "Hello",
    "Who are you?",
    "Tell me about AI",
    "Good Morning",
]

for q in test_queries:
    response = agent(q)
    print("Query:", q)
    print("Response:")
    print(json.dumps(response, indent=4))
    print("-" * 60)

## Interactive Testing

This last part lets you type in your own query and see what the agent does with it live. Type "exit" whenever you want to stop.

In [ ]:
while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.strip().lower() == "exit":
        print("Exiting interactive mode. Goodbye!")
        break
    response = agent(user_input)
    print("Response:")
    print(json.dumps(response, indent=4))

## Results

All 12 test cases gave the response type I expected:

- The valid calculation queries returned `type: "calculation"` with the correct numeric answers.
- The broken calculation (`"calculate abc+15"`) was caught and returned `type: "error"` instead of crashing.
- The keyword queries returned `type: "keywords"` with a list of up to 5 relevant words each time.
- The empty keyword query (just `"keywords"` with nothing after it) correctly came back as `type: "error"`.
- The general queries (greetings, random questions) all correctly fell through to `type: "general"`.

Every single response had exactly the "type" and "result" keys, nothing extra and nothing missing.

## Conclusion

Overall, the agent does what it's supposed to - it reads the query, figures out if it's a calculation, a keyword request, or something general, sends it to the right tool, and always comes back in the same consistent format. Invalid and empty inputs are handled properly too, so the notebook doesn't crash no matter what gets typed in.